# Day 18 · Colab 2 — Multi-tool Agent Chaining + Redis Event Queue

**Agentic Systems Bootcamp**

Here the agent does real **orchestration**: it *chains* tools to place an order —
`check_inventory → create_order → send_confirmation` — mixing **synchronous** tools
(SQLite reads/writes) with an **asynchronous** one (enqueue an email job onto a Redis Stream,
handled later by a separate **worker** via a consumer group).

Runs entirely in Colab: `fakeredis` provides `XADD` / `XREADGROUP`, SQLite is built in.

> **Patterns (from the deck):** *routing* (pick the right tool), *prompt chaining* (feed one tool's
> output into the next), *parallel* reads, and the **sync-vs-async** boundary — slow side-effects go
> on a queue so the agent turn stays fast.

## Step 1 — Install & configure

In [1]:
!pip install -q anthropic fakeredis 2>/dev/null
import os, getpass
if not os.environ.get('ANTHROPIC_API_KEY'):
    try:
        from google.colab import userdata
        os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    except Exception:
        pass
if not os.environ.get('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY (blank = offline mock): ')
LIVE = bool(os.environ.get('ANTHROPIC_API_KEY'))
MODEL = 'claude-sonnet-4-6'
print('LIVE mode' if LIVE else 'OFFLINE mock mode')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 929.8/929.8 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.0/141.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 499.9/499.9 kB 15.4 MB/s eta 0:00:00
LIVE mode


## Step 2 — SQLite orders + inventory database

Two tables: `inventory` (stock per SKU) and `orders` (what we create).
All access is through **parameterised** SQL — never string-formatted — and reads are row-limited.

In [2]:
import sqlite3
db = sqlite3.connect(':memory:', check_same_thread=False)
db.execute('CREATE TABLE inventory (sku TEXT PRIMARY KEY, name TEXT, qty INTEGER, price REAL)')
db.execute('CREATE TABLE orders (id INTEGER PRIMARY KEY AUTOINCREMENT, sku TEXT, qty INTEGER, total REAL, status TEXT)')
db.executemany('INSERT INTO inventory VALUES (?,?,?,?)', [
    ('KB-01', 'Mechanical keyboard', 12, 129.0),
    ('HUB-2', 'USB-C hub',           0,  58.0),
    ('MON-4', '4K monitor',          5, 410.0),
])
db.commit()
print('inventory seeded')

inventory seeded


## Step 3 — A Redis Stream as the email job queue

`send_confirmation` won't send email inline (slow, can fail). It **enqueues** a job with `XADD` and
returns a `job_id` immediately. A **worker** later reads with `XREADGROUP` (a consumer group gives us
at-least-once delivery + acks) and writes the result back to a `jobresult:<id>` key.

In [3]:
import fakeredis, json, time, uuid
r = fakeredis.FakeStrictRedis()
STREAM = 'emails'
GROUP  = 'mailers'
try:
    r.xgroup_create(STREAM, GROUP, id='0', mkstream=True)
except Exception as e:
    print('group exists:', e)

def enqueue_email(to, subject, body):
    job_id = uuid.uuid4().hex[:8]
    r.xadd(STREAM, {'job_id': job_id, 'to': to, 'subject': subject, 'body': body})
    r.set(f'jobresult:{job_id}', 'queued')
    return job_id

print('queue ready; sample job id ->', enqueue_email('a@x.com', 'hi', 'test'))

queue ready; sample job id -> 97610916


## Step 4 — The worker (consumer group)

In production this is a separate process. Here we drain the stream on demand. Each message is
processed, the result stored, and the message **acked** with `XACK` so it isn't redelivered.

In [4]:
def run_worker(max_msgs=10):
    processed = 0
    resp = r.xreadgroup(GROUP, 'worker-1', {STREAM: '>'}, count=max_msgs)
    for _stream, msgs in resp or []:
        for msg_id, fields in msgs:
            f = {k.decode(): v.decode() for k, v in fields.items()}
            # … here you'd call a real email provider …
            r.set(f"jobresult:{f['job_id']}", 'sent')
            r.xack(STREAM, GROUP, msg_id)
            processed += 1
    return processed

print('worker processed', run_worker(), 'job(s)')

worker processed 1 job(s)


## Step 5 — Tool functions (sync + async)

`check_inventory` and `create_order` are **synchronous** — they touch SQLite and return immediately.
`create_order` re-checks stock and decrements it in one transaction (guarding against overselling).
`send_confirmation` is **asynchronous** — it enqueues and returns a `job_id`. `check_job` polls a result.

In [5]:
def check_inventory(sku: str):
    row = db.execute('SELECT sku,name,qty,price FROM inventory WHERE sku=? LIMIT 1', (sku,)).fetchone()
    if not row:
        return {'error': f'unknown sku {sku}'}
    return {'sku': row[0], 'name': row[1], 'qty': row[2], 'price': row[3]}

def create_order(sku: str, qty: int):
    cur = db.execute('SELECT qty,price FROM inventory WHERE sku=? LIMIT 1', (sku,)).fetchone()
    if not cur:
        return {'error': f'unknown sku {sku}'}
    have, price = cur
    if qty <= 0:
        return {'error': 'qty must be positive'}
    if have < qty:
        return {'error': f'insufficient stock: have {have}, need {qty}'}
    db.execute('UPDATE inventory SET qty=qty-? WHERE sku=?', (qty, sku))
    cur2 = db.execute('INSERT INTO orders (sku,qty,total,status) VALUES (?,?,?,?)',
                      (sku, qty, round(price*qty, 2), 'created'))
    db.commit()
    return {'order_id': cur2.lastrowid, 'sku': sku, 'qty': qty, 'total': round(price*qty, 2)}

def send_confirmation(to: str, order_id: int):
    job_id = enqueue_email(to, f'Order {order_id} confirmed', f'Your order {order_id} is on its way.')
    return {'job_id': job_id, 'status': 'queued'}

def check_job(job_id: str):
    v = r.get(f'jobresult:{job_id}')
    return {'job_id': job_id, 'status': v.decode() if v else 'unknown'}

# quick manual chain
inv = check_inventory('KB-01'); print(inv)
od  = create_order('KB-01', 2);  print(od)
jb  = send_confirmation('asha@x.com', od['order_id']); print(jb)
run_worker(); print(check_job(jb['job_id']))

{'sku': 'KB-01', 'name': 'Mechanical keyboard', 'qty': 12, 'price': 129.0}
{'order_id': 1, 'sku': 'KB-01', 'qty': 2, 'total': 258.0}
{'job_id': '1d9ab0f1', 'status': 'queued'}
{'job_id': '1d9ab0f1', 'status': 'sent'}


## Step 6 — Tool schemas + dispatch

In [6]:
TOOLS = [
  {'name':'check_inventory','description':'Check stock and price for a product SKU before ordering.',
   'input_schema':{'type':'object','properties':{'sku':{'type':'string','description':'Product SKU, e.g. KB-01.'}},'required':['sku']}},
  {'name':'create_order','description':'Create an order for a SKU and quantity. Fails if stock is insufficient.',
   'input_schema':{'type':'object','properties':{'sku':{'type':'string'},'qty':{'type':'integer','description':'Units to order (>0).'}},'required':['sku','qty']}},
  {'name':'send_confirmation','description':'Queue a confirmation email for a created order. Returns a job_id immediately.',
   'input_schema':{'type':'object','properties':{'to':{'type':'string','description':'Customer email.'},'order_id':{'type':'integer'}},'required':['to','order_id']}},
  {'name':'check_job','description':'Check the status of a queued email job by job_id.',
   'input_schema':{'type':'object','properties':{'job_id':{'type':'string'}},'required':['job_id']}},
]
DISPATCH = {'check_inventory':check_inventory,'create_order':create_order,
            'send_confirmation':send_confirmation,'check_job':check_job}

def run_tool(name, args):
    fn = DISPATCH.get(name)
    if not fn: return {'error': f'unknown tool {name}'}, True
    try:
        out = fn(**args)
        return out, isinstance(out, dict) and 'error' in out
    except Exception as e:
        return {'error': repr(e)}, True
print('tools ready')

tools ready


## Step 7 — The orchestrating agent loop

Same `stop_reason` loop as Colab 1, but now the model **chains** several tools in one turn and we
drain the worker afterwards. The offline mock scripts the full chain so the notebook runs without a key.

In [7]:
import json
SYSTEM = (
  'You are an ordering agent. To place an order: first check_inventory, then create_order, '
  'then send_confirmation with the returned order_id. Report the order total and the email job status. '
  'If stock is insufficient, say so and do not create the order.'
)

def agent(user_text, max_steps=8, verbose=True):
    if not LIVE:
        if verbose: print('… (mock) chaining check_inventory → create_order → send_confirmation')
        inv,_ = run_tool('check_inventory', {'sku':'MON-4'})
        od,_  = run_tool('create_order', {'sku':'MON-4','qty':1})
        jb,_  = run_tool('send_confirmation', {'to':'asha@x.com','order_id':od['order_id']})
        run_worker()
        st,_  = run_tool('check_job', {'job_id':jb['job_id']})
        return f"(mock) Order {od['order_id']} total ${od['total']}; email {st['status']}."

    from anthropic import Anthropic
    A = Anthropic()
    messages = [{'role':'user','content':user_text}]
    for _ in range(max_steps):
        resp = A.messages.create(model=MODEL, max_tokens=1024, system=SYSTEM, tools=TOOLS, messages=messages)
        if resp.stop_reason == 'tool_use':
            messages.append({'role':'assistant','content':[b.model_dump() for b in resp.content]})
            results = []
            for b in resp.content:
                if b.type == 'tool_use':
                    if verbose: print(f'  → {b.name}({b.input})')
                    out, is_err = run_tool(b.name, b.input)
                    results.append({'type':'tool_result','tool_use_id':b.id,
                                    'content':json.dumps(out),'is_error':is_err})
            messages.append({'role':'user','content':results})
            run_worker()  # drain any queued email jobs between steps
            continue
        return ''.join(b.text for b in resp.content if b.type=='text')
    return '(max steps reached)'

print(agent('Order one 4K monitor (SKU MON-4) and email asha@x.com the confirmation.'))

  → check_inventory({'sku': 'MON-4'})
  → create_order({'sku': 'MON-4', 'qty': 1})
  → send_confirmation({'to': 'asha@x.com', 'order_id': 2})
  → check_job({'job_id': 'e8936be5'})
Everything went through smoothly! Here's a summary:

- **Product:** 4K Monitor (SKU: MON-4)
- **Quantity:** 1
- **Order Total:** $410.00
- **Order ID:** #2
- **Confirmation Email:** Sent to asha@x.com ✅


## Step 8 — Show the resulting state

In [8]:
print('Orders table:')
for row in db.execute('SELECT id,sku,qty,total,status FROM orders').fetchall():
    print(' ', row)
print('Remaining stock:')
for row in db.execute('SELECT sku,qty FROM inventory').fetchall():
    print(' ', row)
# Try an out-of-stock order to see graceful failure
print('Out-of-stock attempt:', run_tool('create_order', {'sku':'HUB-2','qty':1}))

Orders table:
  (1, 'KB-01', 2, 258.0, 'created')
  (2, 'MON-4', 1, 410.0, 'created')
Remaining stock:
  ('KB-01', 10)
  ('HUB-2', 0)
  ('MON-4', 4)
Out-of-stock attempt: ({'error': 'insufficient stock: have 0, need 1'}, True)


## Capstone Task

## Customer Support Agent

Capabilities :
| Tool             | Type           |
| ---------------- | -------------- |
| search_kb        | Vector Search  |
| create_ticket    | Human Approval |
| send_email       | Async Queue    |
| get_order_status | SQLite         |


### Architecture

### Step 1: Install Additional Packages

In [9]:
%%capture
!pip install sentence-transformers faiss-cpu

###Step 2: Redis Short-Term Memory

In [23]:
import json

def save_memory(user_id, role, content):
    key = f"memory:{user_id}"

    r.rpush(
        key,
        json.dumps({
            "role": role,
            "content": content
        })
    )

    r.ltrim(key, -10, -1)


def get_memory(user_id):
    key = f"memory:{user_id}"

    messages = r.lrange(key, 0, -1)

    return [
        json.loads(msg.decode())
        for msg in messages
    ]

### Step 3: Vector Store Long-Term Recall

In [11]:
support_docs = [
    "Refunds take 5 to 7 business days.",
    "Orders can be cancelled before shipment.",
    "Password reset link expires after 24 hours.",
    "Premium members receive priority support.",
    "Shipping usually takes 3 business days."
]

### Step 4: Embeddings

In [12]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

model_embed = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

embeddings = model_embed.encode(
    support_docs
)

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(embeddings).astype("float32")
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Step 5: Tool 1 – Knowledge Search

In [13]:
def search_kb(query):

    query_vector = model_embed.encode(
        [query]
    )

    D, I = index.search(
        np.array(query_vector).astype("float32"),
        2
    )

    results = [
        support_docs[i]
        for i in I[0]
    ]

    return "\n".join(results)

### Step 6: Tool 2 – Order Status

In [14]:
def get_order_status(order_id):

    conn = sqlite3.connect("orders.db")

    cursor = conn.cursor()

    cursor.execute(
        """
        SELECT * FROM orders
        WHERE order_id=?
        """,
        (order_id,)
    )

    row = cursor.fetchone()

    conn.close()

    return str(row)

### Step 7: Tool 3 – Create Ticket (Human Approval)

In [15]:
def create_support_ticket(
        user_name,
        issue
):

    approval = input(
        f"""
Approve Ticket?

User: {user_name}

Issue:
{issue}

Type YES:
"""
    )

    if approval != "YES":
        return "Ticket rejected"

    ticket_id = f"TKT-{int(time.time())}"

    return {
        "ticket_id": ticket_id,
        "status": "created"
    }

### Step 8: Tool 4 – Async Email Queue

In [24]:
def send_support_email(email, message):

    event_id = r.xadd(
        STREAM,
        {
            "email": email,
            "message": message
        }
    )

    return str(event_id)

### Step 9: Add Retry Decorator

In [17]:
from functools import wraps
import time

def retry(
        retries=3,
        delay=1
):

    def decorator(func):

        @wraps(func)
        def wrapper(*args, **kwargs):

            for attempt in range(retries):

                try:
                    return func(
                        *args,
                        **kwargs
                    )

                except Exception as e:

                    print(
                        f"Retry {attempt+1}"
                    )

                    time.sleep(delay)

            raise Exception(
                "Max retries exceeded"
            )

        return wrapper

    return decorator

### Step 10: Tracing

In [18]:
trace_logs = []

def trace(step, data):

    trace_logs.append({
        "time": time.time(),
        "step": step,
        "data": str(data)
    })

    print(
        f"[TRACE] {step}: {data}"
    )

### Step 11: Prompt Cache

In [25]:
prompt_cache = {}

def get_cached_response(user_input):

    if user_input in prompt_cache:
        print("✅ Cache Hit")
        return prompt_cache[user_input]

    print("❌ Cache Miss")

    response = agent(user_input)

    prompt_cache[user_input] = response

    return response

In [28]:
save_memory(
    "user1",
    "user",
    "What is refund policy?"
)

save_memory(
    "user1",
    "assistant",
    "Refunds take 5 to 7 days."
)

print(get_memory("user1"))

[{'role': 'user', 'content': 'What is refund policy?'}, {'role': 'assistant', 'content': 'Refunds take 5 to 7 days.'}, {'role': 'user', 'content': 'What is refund policy?'}, {'role': 'assistant', 'content': 'Refunds take 5 to 7 days.'}]


### Architecture

# Customer Support Agent

## Overview

This project extends the Lab-2 Order Processing Agent into a Customer Support Agent.

## Features

### Short-Term Memory
Redis stores recent conversation history.

### Long-Term Recall
FAISS vector database stores support knowledge articles and performs semantic retrieval.

### Tools

1. search_kb
2. get_order_status
3. create_support_ticket
4. send_support_email

### Human Approval

Support ticket creation requires manual approval before execution.

### Async Processing

Email notifications are pushed to Redis Streams and processed by a worker asynchronously.

### Reliability

- Retry decorator
- Prompt cache
- Trace logging

## Flow

User
→ Agent
→ Memory Lookup
→ Tool Selection
→ Human Approval (if needed)
→ Async Queue
→ Worker
→ Response